In [1]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, 'patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio_from_raw70m.csv'), index_col=0)
        # cell_label = cell_label.apply(lambda x: x*100.0)
        
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        label_index_set = set(label_df.index)
        patch_index_set = set(patch_list)
        and_set = label_index_set & patch_index_set

        patch_list = list(and_set)
        self.label_df = label_df.loc[patch_list]
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, 'patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        label = self.label_df.loc[patch_id].values
        label = torch.Tensor(label)

        return patch_id, data, label

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [2]:
# prepare necessary arguments and WSI sample list

tif_list = '/data1/r20user3/shared_project/Hist2Cell/data/stnet'
tif_list = glob(tif_list + '/*[!xlsx|ipynb]')
tif_list.sort()
tif_list

['/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_D2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_E1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_E2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_D2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_E1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_E2',
 '/data1/r20user3/shared_

In [3]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['23209_C1',
 '23209_C2',
 '23209_D1',
 '23268_C1',
 '23268_C2',
 '23268_D1',
 '23269_C1',
 '23269_C2',
 '23269_D1',
 '23270_D2',
 '23270_E1',
 '23270_E2',
 '23272_D2',
 '23272_E1',
 '23272_E2',
 '23277_D2',
 '23277_E1',
 '23277_E2',
 '23287_C1',
 '23287_C2',
 '23287_D1',
 '23288_D2',
 '23288_E1',
 '23288_E2',
 '23377_C1',
 '23377_C2',
 '23377_D1',
 '23450_D2',
 '23450_E1',
 '23450_E2',
 '23506_C1',
 '23506_C2',
 '23506_D1',
 '23508_D2',
 '23508_E1',
 '23508_E2',
 '23567_D2',
 '23567_E1',
 '23567_E2',
 '23803_D2',
 '23803_E1',
 '23803_E2',
 '23810_D2',
 '23810_E1',
 '23810_E2',
 '23895_C1',
 '23895_C2',
 '23895_D1',
 '23901_C2',
 '23901_D1',
 '23903_C1',
 '23903_C2',
 '23903_D1',
 '23944_D2',
 '23944_E1',
 '23944_E2',
 '24044_D2',
 '24044_E1',
 '24044_E2',
 '24105_C1',
 '24105_C2',
 '24105_D1',
 '24220_D2',
 '24220_E1',
 '24220_E2',
 '24223_D2',
 '24223_E1',
 '24223_E2']

In [4]:
# prepare my GPU

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [6]:
batch_size = 512
patch_dir = "/data1/r20user3/shared_project/Hist2Cell/data/stnet"

graphs = dict()
for slide in test_slides:
    print(slide)
    test_data = PTC_cell(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)

    spots_coord = pd.read_csv(os.path.join("/data1/r20user3/shared_project/Hist2Cell/data/stnet", slide, "spots.csv"), index_col=0)

    with torch.no_grad():
        # model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []

        for name, data, label in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            test_label_array.append(label.cpu().detach().numpy())
            
            # features, output = model(data)
            # test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            test_prediction_array.append(data.cpu().detach().numpy())

    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    for i in range(len(test_label_array)):
        if len(test_label_array[i].shape) <= 1:
            test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_prediction_array = np.concatenate(test_prediction_array)
    test_label_array = np.concatenate(test_label_array)
    
    test_names = list()
    for names in test_name_array:
        test_names = test_names+names
    test_node_x_y = list()
    for item in test_names:
        test_node_x_y.append([int(item.split('x')[0]), int(item.split('x')[1])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 2 or x2 >= x1 + 2 or y2 <= y1 - 2 or y2 >= y1 + 2:
                    continue
                else:
                    adj[i][j] = 1.0

    graphs[slide] = {
        'features': test_prediction_array,
        'labels': test_label_array,
        'adj': adj,
        'names': test_names,
        'coord': spots_coord.loc[test_names].values,
    }

    print("OK")

23209_C1
OK
23209_C2
OK
23209_D1
OK
23268_C1
OK
23268_C2
OK
23268_D1
OK
23269_C1
OK
23269_C2
OK
23269_D1
OK
23270_D2
OK
23270_E1
OK
23270_E2
OK
23272_D2
OK
23272_E1
OK
23272_E2
OK
23277_D2
OK
23277_E1
OK
23277_E2
OK
23287_C1
OK
23287_C2
OK
23287_D1
OK
23288_D2
OK
23288_E1
OK
23288_E2
OK
23377_C1
OK
23377_C2
OK
23377_D1
OK
23450_D2
OK
23450_E1
OK
23450_E2
OK
23506_C1
OK
23506_C2
OK
23506_D1
OK
23508_D2
OK
23508_E1
OK
23508_E2
OK
23567_D2
OK
23567_E1
OK
23567_E2
OK
23803_D2
OK
23803_E1
OK
23803_E2
OK
23810_D2
OK
23810_E1
OK
23810_E2
OK
23895_C1
OK
23895_C2
OK
23895_D1
OK
23901_C2
OK
23901_D1
OK
23903_C1
OK
23903_C2
OK
23903_D1
OK
23944_D2
OK
23944_E1
OK
23944_E2
OK
24044_D2
OK
24044_E1
OK
24044_E2
OK
24105_C1
OK
24105_C2
OK
24105_D1
OK
24220_D2
OK
24220_E1
OK
24220_E2
OK
24223_D2
OK
24223_E1
OK
24223_E2
OK


In [7]:
graphs['23209_C1']['features'].shape

(294, 3, 224, 224)

In [8]:
graphs['23209_C1']['coord'].shape

(294, 2)

In [9]:
graphs['23209_C1']['names']

['9x26',
 '9x19',
 '14x13',
 '6x17',
 '17x14',
 '16x9',
 '13x20',
 '4x17',
 '9x24',
 '24x16',
 '7x18',
 '9x15',
 '19x16',
 '20x18',
 '11x24',
 '15x9',
 '7x15',
 '17x17',
 '6x12',
 '4x12',
 '7x20',
 '19x8',
 '3x16',
 '19x5',
 '19x19',
 '11x12',
 '12x23',
 '9x12',
 '6x24',
 '13x15',
 '2x15',
 '13x18',
 '6x25',
 '21x17',
 '19x9',
 '7x19',
 '5x22',
 '16x10',
 '23x14',
 '13x11',
 '16x14',
 '16x7',
 '3x20',
 '18x8',
 '21x9',
 '8x17',
 '17x12',
 '4x15',
 '16x18',
 '14x21',
 '4x22',
 '15x16',
 '15x19',
 '20x19',
 '11x13',
 '7x17',
 '16x15',
 '20x7',
 '14x12',
 '5x20',
 '10x19',
 '14x17',
 '20x15',
 '20x9',
 '20x11',
 '10x12',
 '17x20',
 '19x17',
 '13x21',
 '13x17',
 '7x22',
 '19x10',
 '14x10',
 '11x16',
 '8x12',
 '11x21',
 '9x20',
 '15x21',
 '12x19',
 '4x21',
 '14x16',
 '13x16',
 '13x12',
 '2x18',
 '19x12',
 '21x10',
 '3x18',
 '5x16',
 '6x19',
 '8x26',
 '12x9',
 '19x15',
 '21x15',
 '18x16',
 '21x14',
 '22x16',
 '12x11',
 '17x15',
 '22x12',
 '17x9',
 '16x11',
 '23x16',
 '3x22',
 '14x20',
 '20x6

In [10]:
graph = graphs
for slide in graph:
    print(slide)

23209_C1
23209_C2
23209_D1
23268_C1
23268_C2
23268_D1
23269_C1
23269_C2
23269_D1
23270_D2
23270_E1
23270_E2
23272_D2
23272_E1
23272_E2
23277_D2
23277_E1
23277_E2
23287_C1
23287_C2
23287_D1
23288_D2
23288_E1
23288_E2
23377_C1
23377_C2
23377_D1
23450_D2
23450_E1
23450_E2
23506_C1
23506_C2
23506_D1
23508_D2
23508_E1
23508_E2
23567_D2
23567_E1
23567_E2
23803_D2
23803_E1
23803_E2
23810_D2
23810_E1
23810_E2
23895_C1
23895_C2
23895_D1
23901_C2
23901_D1
23903_C1
23903_C2
23903_D1
23944_D2
23944_E1
23944_E2
24044_D2
24044_E1
24044_E2
24105_C1
24105_C2
24105_D1
24220_D2
24220_E1
24220_E2
24223_D2
24223_E1
24223_E2


In [11]:
from torch import Tensor
import torch
import os

for slide_name in graph:
    x = Tensor(graph[slide_name]['features'])
    from torch_geometric.utils import dense_to_sparse
    adj = Tensor(graph[slide_name]['adj'])
    edge_index, _ = dense_to_sparse(adj)
    y = Tensor(graph[slide_name]['labels'])
    from torch_geometric.data import Data
    pos = Tensor(graph[slide_name]['coord'])

    data = Data(x=x, edge_index=edge_index, y=y, pos=pos)
    
    torch.save(data, os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/stnet_from_raw70m", slide_name+".pt"))

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import torch

data1 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/stnet_from_raw70m/24223_E1.pt")
data2 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/stnet_from_raw70m/24223_E2.pt")

In [13]:
from torch_geometric.data import Batch

data = Batch.from_data_list([data1, data2])
data

DataBatch(x=[725, 3, 224, 224], edge_index=[2, 5957], y=[725, 289], pos=[725, 2], batch=[725], ptr=[3])

In [14]:
from random import shuffle
from torch_geometric.loader import NeighborLoader
import torch_geometric

torch_geometric.typing.WITH_PYG_LIB = False

from torch_geometric.seed import seed_everything
seed_everything(0)

loader = NeighborLoader(
    data,
    # Sample 30 neighbors for each node for 2 iterations
    num_neighbors=[-1, -1],
    # Use a batch size of 128 for sampling training nodes
    batch_size=1,
    directed=False,
    input_nodes=None,
    # disjoint=True,
    shuffle=True
)

i = 0
for sampled_data in loader:
    print(sampled_data.input_id)
    i = i + 1
    if i > 10:
        break

tensor([51])
tensor([712])
tensor([317])
tensor([53])
tensor([635])
tensor([146])
tensor([642])
tensor([466])
tensor([43])
tensor([373])
tensor([191])


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "


In [15]:
sampled_data

DataBatch(x=[11, 3, 224, 224], edge_index=[2, 61], y=[11, 289], pos=[11, 2], ptr=[3], n_id=[11], e_id=[61], input_id=[1], batch_size=1)

In [16]:
sampled_data.pos

tensor([[6419.8691, 6963.8149],
        [6098.8350, 6653.7148],
        [6090.2578, 7232.3979],
        [6099.4961, 6942.4058],
        [6395.0850, 6670.9390],
        [5795.7949, 6662.0229],
        [5802.8159, 6967.5840],
        [5795.6909, 6371.5518],
        [6092.6421, 6360.1118],
        [6410.7749, 6386.8711],
        [6715.9502, 6369.4971]])